In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('../../../src/zero_to_hero/part_02_makemore/names.txt', 'r').read().splitlines()

In [3]:
len(words)

32033

In [4]:
# We had S/E special tokens, but now just "."
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [30]:
# ------------ now made respectable :) ---------------

In [27]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words, block_size):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1], block_size)
Xdev, Ydev = build_dataset(words[n1:n2], block_size)
Xte, Yte = build_dataset(words[n2:], block_size)

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [98]:
# Individual iterations take smaller
# In practice - people use minibatches - only updating using some small dataset (basically bagging?)

In [17]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
  p.requires_grad = True

In [18]:

# minibatch construction
# This means gradient is noisy, but unbiased

mb_size = 100
for _ in range(1000):
    #minibatch
    ix = torch.randint(0, Xtr.shape[0], (mb_size,))
    # forward pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits,Ytr[ix])
    print(loss.item())
    # Backward pass - should be none!
    for p in parameters:
        p.grad = None
    
    loss.backward()
    for p in parameters:
        p.data += -0.1 * p.grad

21.75513458251953
17.076065063476562
16.507373809814453
15.106703758239746
14.730792045593262
13.124776840209961
12.173044204711914
12.385112762451172
10.806832313537598
11.281390190124512
10.3631591796875
9.659891128540039
9.884000778198242
9.882535934448242
10.867349624633789
9.2952299118042
8.80380630493164
8.725615501403809
8.297011375427246
9.598616600036621
9.518502235412598
8.443561553955078
7.8341569900512695
7.330626964569092
7.9330058097839355
7.480666637420654
7.30726432800293
8.377089500427246
7.961971282958984
7.622998237609863
5.968314170837402
6.829749584197998
4.9660115242004395
6.123058319091797
5.641962051391602
6.5931596755981445
5.690673351287842
6.626424789428711
6.88753604888916
5.99533748626709
5.878497123718262
5.498652458190918
5.926759719848633
5.915703296661377
5.55126953125
5.095653533935547
5.177252769470215
4.913369178771973
5.094287395477295
5.259512424468994
4.6306562423706055
5.101034164428711
5.136800765991211
4.944989204406738
5.860584735870361
4.8133

In [19]:
    emb = C[Xtr]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits,Ytr)
    print(loss)

tensor(2.6008, grad_fn=<NllLossBackward0>)


In [86]:
import optuna
def objective(trial: optuna.Trial, Xtr, Ytr) -> float:
    # Add weight decay?
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 1e-0, log=True),
        "n_embedding": trial.suggest_int("n_embedding", 2, 10),
        "mb_size": trial.suggest_categorical("mb_size", [32, 64, 128, 256, 512, 1024]),
        "n_hidden": trial.suggest_int("n_hidden", 50, 1000),
        "reg": trial.suggest_float("reg", 1e-5, 1e-1, log=True),
        "steps": trial.suggest_int("steps", 10000, 100000)
    }

    loss, tuned_params = compute_loss(params, Xtr, Ytr)
    return loss

In [87]:
def compute_loss(params, Xtr, Ytr):
    mb_size = params["mb_size"]
    n_embedding = params["n_embedding"]
    #n_context = params["n_context"]
    n_context = 3
    learning_rate = params["learning_rate"]
    weight_decay_lambda = params["reg"]
    steps = params["steps"]

    # Now define params
    g = torch.Generator().manual_seed(2147483647) # for reproducibility
    C = torch.randn((27, n_embedding), generator=g)
    W1 = torch.randn((n_context * n_embedding, n_hidden), generator=g)
    b1 = torch.randn(n_hidden, generator=g)
    W2 = torch.randn((n_hidden, 27), generator=g)
    b2 = torch.randn(27, generator=g)
    parameters = [C, W1, b1, W2, b2]
    steps = 10000
    for p in parameters:
        p.requires_grad = True
    
    for i in range(steps):
        #full batch
        ix = torch.randint(0, Xtr.shape[0], (mb_size,))
        # forward pass
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits,Ytr[ix]) + weight_decay_lambda * (W1**2).sum() + weight_decay_lambda * (W2**2).sum()
        #print(loss.item())
        # Backward pass - should be none!
        for p in parameters:
            p.grad = None
        
        loss.backward()
        lr = learning_rate
        for p in parameters:
            p.data += -lr * p.grad

    small_steps = int(steps / 5)
    for i in range(small_steps):
        #full batch
        ix = torch.randint(0, Xtr.shape[0], (mb_size,))
        # forward pass
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits,Ytr[ix]) + weight_decay_lambda * (W1**2).sum() + weight_decay_lambda * (W2**2).sum()
        #print(loss.item())
        # Backward pass - should be none!
        for p in parameters:
            p.grad = None
        
        loss.backward()
        lr = learning_rate / 10
        for p in parameters:
            p.data += -lr * p.grad

    emb = C[Xtr]
    h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1) 
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr)

    tuned_params = [C, W1, b1, W2, b2]
    return loss, tuned_params

In [88]:
from functools import partial
study = optuna.create_study(direction="minimize")  # use "maximize" if optimizing a metric like accuracy
study.optimize(partial(objective, Xtr=Xtr, Ytr=Ytr), n_trials=100, timeout=None)  # adjust n_trials/timeout as needed

[I 2026-06-30 16:06:53,003] A new study created in memory with name: no-name-4397497b-2a71-4f23-972c-c09f04a07e82
[I 2026-06-30 16:06:56,346] Trial 0 finished with value: 2.4667935371398926 and parameters: {'learning_rate': 0.020721574096143523, 'n_embedding': 3, 'mb_size': 64, 'n_hidden': 240, 'reg': 1.2445790219393435e-05, 'steps': 97246}. Best is trial 0 with value: 2.4667935371398926.
[I 2026-06-30 16:07:02,675] Trial 1 finished with value: 2.229818344116211 and parameters: {'learning_rate': 0.3883322387497778, 'n_embedding': 4, 'mb_size': 512, 'n_hidden': 591, 'reg': 2.334085198305022e-05, 'steps': 56257}. Best is trial 1 with value: 2.229818344116211.
[I 2026-06-30 16:07:08,203] Trial 2 finished with value: 2.5986599922180176 and parameters: {'learning_rate': 0.03266787421382829, 'n_embedding': 8, 'mb_size': 256, 'n_hidden': 349, 'reg': 0.04154942627910372, 'steps': 39592}. Best is trial 1 with value: 2.229818344116211.
[I 2026-06-30 16:07:14,454] Trial 3 finished with value: 2.4

In [89]:
study.best_trial.params

{'learning_rate': 0.9954213334422817,
 'n_embedding': 8,
 'mb_size': 1024,
 'n_hidden': 454,
 'reg': 8.13775452119088e-05,
 'steps': 14386}

In [90]:
params = study.best_trial.params
_, tuned_params = compute_loss(params, Xtr, Ytr)

In [91]:
C, W1, b1, W2, b2 = tuned_params
n_embedding = params['n_embedding']
n_context = 3

In [ ]:
# Really I should fold in n_context, validate on dev and evaluate both on test
# And do proper learning rate decay but hey we're done

In [92]:
emb = C[Xtr] # (32, 3, 2)
h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ytr)
loss

tensor(2.0721, grad_fn=<NllLossBackward0>)

In [93]:
emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

tensor(2.1134, grad_fn=<NllLossBackward0>)